In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

engine = create_engine(os.getenv("DATABASE_URL"))

In [2]:
import pandas as pd
import numpy as np

# 1. Load customer_360

In [3]:
ci = pd.read_sql(
    "SELECT * FROM customer_360",
    engine
)

print("customer_360:", ci.shape)

customer_360: (1200, 30)


# 2. Clean fields used by the decision layer


In [4]:
ci["ticket_count"] = ci["ticket_count"].fillna(0)
ci["high_priority_tickets"] = ci["high_priority_tickets"].fillna(0)
ci["urgent_tickets"] = ci["urgent_tickets"].fillna(0)
ci["unresolved_tickets"] = ci["unresolved_tickets"].fillna(0)

ci["mrr"] = ci["mrr"].fillna(0)
ci["open_amount"] = ci["open_amount"].fillna(0)
ci["uncollectible_amount"] = ci["uncollectible_amount"].fillna(0)

# 3. Recreate the six risk signals

In [5]:
ci["low_engagement"] = (
    ci["user_count"] <= 5
).astype(int)

ci["low_mrr"] = (
    ci["mrr"] < ci["mrr"].median()
).astype(int)

ci["payment_risk"] = (
    (ci["open_amount"] > 0) |
    (ci["uncollectible_amount"] > 0)
).astype(int)

ci["support_pressure"] = (
    (ci["high_priority_tickets"] > 0) |
    (ci["urgent_tickets"] > 0)
).astype(int)

ci["unresolved_risk"] = (
    ci["unresolved_tickets"] > 0
).astype(int)

ci["low_satisfaction"] = (
    ci["avg_satisfaction"].fillna(
        ci["avg_satisfaction"].median()
    ) < 3.5
).astype(int)

In [6]:
ci.head(5)

,account_id,company_name,industry,country,employee_count,signup_date,plan,user_count,admin_count,member_count,...,urgent_tickets,avg_satisfaction,avg_resolution_hours,unresolved_tickets,low_engagement,low_mrr,payment_risk,support_pressure,unresolved_risk,low_satisfaction
0,115359,JetMetrics Software,Financial Services,Netherlands,23,2023-06-21 10:57:18,Starter,3,0,3,...,1.0,4.29,29.77,0.0,1,1,0,1,0,0
1,596031,LaunchPad HQ Holdings,Healthcare,Australia,12,2024-03-01 23:05:21,Growth,11,3,8,...,0.0,5.00,24.74,1.0,0,0,1,1,1,0
2,550131,NodeSync Software,Retail,Germany,91,2024-06-13 00:12:52,Growth,35,3,23,...,0.0,3.80,52.10,2.0,0,0,1,1,1,0
3,290736,Relay Systems,Manufacturing,Canada,188,2024-02-06 07:47:11,Enterprise,108,28,61,...,1.0,4.00,16.80,0.0,0,0,1,1,0,0
4,154626,EdgeFlow Technologies Systems 1,Healthcare,India,64,2023-12-09 14:44:55,Growth,20,7,12,...,0.0,5.00,68.70,0.0,0,0,1,1,0,0


# 4. Health score


In [7]:
risk_columns = [
    "low_engagement",
    "low_mrr",
    "payment_risk",
    "support_pressure",
    "unresolved_risk",
    "low_satisfaction"
]

ci["risk_points"] = ci[risk_columns].sum(axis=1)

ci["health_score"] = (100 -(ci["risk_points"]/ len(risk_columns)* 100)).round(0)

In [8]:
ci.columns

Index(['account_id', 'company_name', 'industry', 'country', 'employee_count',
       'signup_date', 'plan', 'user_count', 'admin_count', 'member_count',
       'viewer_count', 'seats', 'subscription_status', 'mrr', 'started_on',
       'ended_on', 'users_per_seat', 'subscription_duration', 'invoice_count',
       'total_invoiced', 'avg_invoice_amount', 'paid_amount', 'open_amount',
       'uncollectible_amount', 'ticket_count', 'high_priority_tickets',
       'urgent_tickets', 'avg_satisfaction', 'avg_resolution_hours',
       'unresolved_tickets', 'low_engagement', 'low_mrr', 'payment_risk',
       'support_pressure', 'unresolved_risk', 'low_satisfaction',
       'risk_points', 'health_score'],
      dtype='str')

In [9]:
ci.head(5)

,account_id,company_name,industry,country,employee_count,signup_date,plan,user_count,admin_count,member_count,...,avg_resolution_hours,unresolved_tickets,low_engagement,low_mrr,payment_risk,support_pressure,unresolved_risk,low_satisfaction,risk_points,health_score
0,115359,JetMetrics Software,Financial Services,Netherlands,23,2023-06-21 10:57:18,Starter,3,0,3,...,29.77,0.0,1,1,0,1,0,0,3,50.0
1,596031,LaunchPad HQ Holdings,Healthcare,Australia,12,2024-03-01 23:05:21,Growth,11,3,8,...,24.74,1.0,0,0,1,1,1,0,3,50.0
2,550131,NodeSync Software,Retail,Germany,91,2024-06-13 00:12:52,Growth,35,3,23,...,52.10,2.0,0,0,1,1,1,0,3,50.0
3,290736,Relay Systems,Manufacturing,Canada,188,2024-02-06 07:47:11,Enterprise,108,28,61,...,16.80,0.0,0,0,1,1,0,0,2,67.0
4,154626,EdgeFlow Technologies Systems 1,Healthcare,India,64,2023-12-09 14:44:55,Growth,20,7,12,...,68.70,0.0,0,0,1,1,0,0,2,67.0


# 5. Health segment


In [10]:
ci["health_segment"] = pd.cut(
    ci["health_score"],
    bins=[-1, 39, 59, 79, 100],
    labels=[
        "Critical",
        "At Risk",
        "Watch",
        "Healthy"
    ]
)

In [11]:
ci.head(5)

,account_id,company_name,industry,country,employee_count,signup_date,plan,user_count,admin_count,member_count,...,unresolved_tickets,low_engagement,low_mrr,payment_risk,support_pressure,unresolved_risk,low_satisfaction,risk_points,health_score,health_segment
0,115359,JetMetrics Software,Financial Services,Netherlands,23,2023-06-21 10:57:18,Starter,3,0,3,...,0.0,1,1,0,1,0,0,3,50.0,At Risk
1,596031,LaunchPad HQ Holdings,Healthcare,Australia,12,2024-03-01 23:05:21,Growth,11,3,8,...,1.0,0,0,1,1,1,0,3,50.0,At Risk
2,550131,NodeSync Software,Retail,Germany,91,2024-06-13 00:12:52,Growth,35,3,23,...,2.0,0,0,1,1,1,0,3,50.0,At Risk
3,290736,Relay Systems,Manufacturing,Canada,188,2024-02-06 07:47:11,Enterprise,108,28,61,...,0.0,0,0,1,1,0,0,2,67.0,Watch
4,154626,EdgeFlow Technologies Systems 1,Healthcare,India,64,2023-12-09 14:44:55,Growth,20,7,12,...,0.0,0,0,1,1,0,0,2,67.0,Watch


# 6. Active revenue exposure


In [12]:
ci["active_at_risk"] = (
    (ci["subscription_status"] == "active") &
    (ci["risk_points"] >= 2)
).astype(int)

ci["active_revenue_at_risk"] = np.where(
    ci["active_at_risk"] == 1,
    ci["mrr"],
    0
)

# 7

In [13]:
def generate_reasons(row):

    reasons = []

    if row["low_engagement"] == 1:
        reasons.append("Low customer engagement")

    if row["low_mrr"] == 1:
        reasons.append("Lower commercial value")

    if row["payment_risk"] == 1:
        reasons.append("Payment / billing risk")

    if row["support_pressure"] == 1:
        reasons.append("High-priority support activity")

    if row["unresolved_risk"] == 1:
        reasons.append("Unresolved support issues")

    if row["low_satisfaction"] == 1:
        reasons.append("Low customer satisfaction")

    if not reasons:
        reasons.append("No major risk signal detected")

    return reasons


ci["risk_reasons"] = ci.apply(
    generate_reasons,
    axis=1
)

ci["risk_reason_text"] = ci["risk_reasons"].apply(
    lambda x: " | ".join(x)
)

ci["risk_reason_count"] = (
    ci["risk_reasons"].apply(len)
)

# 8. Retention priority

In [14]:
def priority(row):

    if row["subscription_status"] != "active":
        return "Not Active"

    if row["risk_points"] >= 4:
        return "Immediate Intervention"

    if row["risk_points"] >= 3:
        return "High Priority"

    if row["risk_points"] == 2:
        return "Monitor"

    return "Low Priority"


ci["retention_priority"] = ci.apply(
    priority,
    axis=1
)

# 9. Recommended business action

In [15]:
def recommended_action(row):

    if row["subscription_status"] != "active":
        return "Historical customer — analyse churn reason"

    if row["payment_risk"] == 1:
        return "Billing intervention"

    if row["low_engagement"] == 1:
        return "Product adoption / onboarding intervention"

    if row["low_satisfaction"] == 1:
        return "Customer success follow-up"

    if row["support_pressure"] == 1:
        return "Support escalation"

    if row["unresolved_risk"] == 1:
        return "Resolve outstanding support issues"

    if row["low_mrr"] == 1:
        return "Engagement and value review"

    return "Routine customer monitoring"


ci["recommended_action"] = ci.apply(
    recommended_action,
    axis=1
)

# 10. Final customer intelligence table

In [16]:
final_columns = [
    "account_id",
    "company_name",
    "industry",
    "country",
    "plan",
    "subscription_status",
    "mrr",
    "seats",
    "user_count",
    "users_per_seat",
    "ticket_count",
    "avg_satisfaction",
    "avg_resolution_hours",
    "open_amount",
    "uncollectible_amount",
    "risk_points",
    "health_score",
    "health_segment",
    "risk_reason_count",
    "risk_reason_text",
    "retention_priority",
    "recommended_action",
    "active_at_risk",
    "active_revenue_at_risk"
]

final_ci = ci[final_columns].copy()

# 11. Save to PostgreSQL

In [17]:
final_ci.to_sql(
    "customer_intelligence",
    engine,
    if_exists="replace",
    index=False
)

print("\nCustomer_intelligence updated successfully.")


Customer_intelligence updated successfully.


# 12. Retention action summary

In [18]:
action_summary = (
    final_ci[
        final_ci["subscription_status"] == "active"
    ]
    .groupby("recommended_action")
    .agg(
        customers=("account_id", "count"),
        mrr=("mrr", "sum")
    )
    .sort_values("mrr", ascending=False)
)

print("\n==============================")
print("RETENTION ACTION SUMMARY")
print("==============================")

print(
    action_summary.round(2).to_string()
)


RETENTION ACTION SUMMARY
                                            customers       mrr
recommended_action                                             
Billing intervention                              624  580823.0
Support escalation                                171  166424.0
Customer success follow-up                         44   80162.0
Routine customer monitoring                        30   30060.0
Resolve outstanding support issues                 17   13592.0
Product adoption / onboarding intervention        134    6828.0
Engagement and value review                        10    1656.0


# 13. Priority summary

In [19]:
priority_summary = (
    final_ci[
        final_ci["subscription_status"] == "active"
    ]
    .groupby("retention_priority")
    .agg(
        customers=("account_id", "count"),
        mrr=("mrr", "sum")
    )
    .sort_values("mrr", ascending=False)
)

print("\n==============================")
print("RETENTION PRIORITY")
print("==============================")

print(
    priority_summary.round(2).to_string()
)


RETENTION PRIORITY
                        customers       mrr
retention_priority                         
Monitor                       292  411322.0
Low Priority                  208  271567.0
High Priority                 267  169380.0
Immediate Intervention        263   27276.0


# 14. Top customers for intervention

In [20]:
top_priority = (
    final_ci[
        (final_ci["subscription_status"] == "active") &
        (
            final_ci["retention_priority"].isin([
                "Immediate Intervention",
                "High Priority"
            ])
        )
    ]
    .sort_values(
        ["mrr", "risk_points"],
        ascending=[False, False]
    )
    .head(20)
)

print("\n==============================")
print("TOP PRIORITY CUSTOMERS")
print("==============================")

print(
    top_priority[
        [
            "account_id",
            "company_name",
            "plan",
            "mrr",
            "health_score",
            "health_segment",
            "risk_points",
            "risk_reason_text",
            "retention_priority",
            "recommended_action"
        ]
    ].to_string(index=False)
)


TOP PRIORITY CUSTOMERS
 account_id              company_name       plan     mrr  health_score health_segment  risk_points                                                                                                risk_reason_text     retention_priority         recommended_action
     662747        ZeroFriction Group Enterprise 18012.0          50.0        At Risk            3                             Payment / billing risk | High-priority support activity | Low customer satisfaction          High Priority       Billing intervention
     757620    GridLogic Collective 1 Enterprise 15958.0          50.0        At Risk            3                             Payment / billing risk | High-priority support activity | Low customer satisfaction          High Priority       Billing intervention
     651891       Lattice HQ Networks Enterprise 12166.0          50.0        At Risk            3                             Payment / billing risk | High-priority support activity | Unresolv

# 15. Tableau files

Export the final customer intelligence dataset for Tableau.

In [21]:
final_ci.to_csv(
    "customer_intelligence.csv",
    index=False
)

print("Tableau file exported: customer_intelligence.csv")
print(final_ci.shape)

Tableau file exported: customer_intelligence.csv
(1200, 24)
